In [1]:
import torch
from einops import rearrange
from llm.nn import scaled_dot_product_attention

In [2]:
# h, num heads
# d_model, embedding dim
# d_k = d_v = d_model / h
# Q.shape -> (... seq_len h*d_k)
# K.shape -> (... seq_len h*d_k)
# V.shape -> (... seq_len h*d_v)
# Wq.shape -> h*d_k, d_model
# Wk.shape -> h*d_k, d_model
# Wv.shape -> h*d_v, d_model
# Wo.shape -> d_model, h*d_v
# MultiHead(𝑄, 𝐾, 𝑉 ) = Concat(head_1 , …, head_h )
#     for head_𝑖 = Attention(𝑄𝑖 , 𝐾𝑖 , 𝑉𝑖 )
# MultiHeadSelfAttention(𝑥) = Wo MultiHead(𝑊𝑄 𝑥, 𝑊𝐾 𝑥, 𝑊𝑉 𝑥)
# Qi.shape -> (... seq_len d_k) 
# x.shape -> (... seq_len d_model)

In [10]:
batch_size = 1
seq_len = 5
d_model = 32
h = 2
d_k = d_v = d_model // h

In [16]:
x = torch.randn(batch_size, seq_len, d_model)
# print(x)
Wq = torch.randn(1, h*d_k, d_model)
Wk = torch.randn(1, h*d_k, d_model)
Wv = torch.randn(1, h*d_v, d_model)

x_t = rearrange(x, 'batch_size seq_len d_model -> batch_size d_model seq_len')

Wq.shape, x_t.shape

(torch.Size([1, 32, 32]), torch.Size([1, 32, 5]))

In [27]:
Q = Wq @ x_t
K = Wk @ x_t
V = Wv @ x_t

Q = rearrange(Q, 'batch_size (d_k num_heads) seq_len -> batch_size num_heads seq_len d_k', num_heads=h)
K = rearrange(K, 'batch_size (d_k num_heads) seq_len -> batch_size num_heads seq_len d_k', num_heads=h)
V = rearrange(V, 'batch_size (d_k num_heads) seq_len -> batch_size num_heads seq_len d_k', num_heads=h)

print(f"{Q.shape = }")
print(f"{K.shape = }")
print(f"{V.shape = }")

Q.shape = torch.Size([1, 2, 5, 16])
K.shape = torch.Size([1, 2, 5, 16])
V.shape = torch.Size([1, 2, 5, 16])


In [35]:
Wo = torch.randn(batch_size, d_model, d_model)
Wo.shape, Wo

(torch.Size([1, 32, 32]),
 tensor([[[-0.8678, -2.0687,  0.8783,  ...,  0.8992,  0.0447, -0.4222],
          [ 0.7856, -0.2031, -0.2220,  ...,  0.9088,  0.2186, -1.1714],
          [-0.8657, -1.8815,  0.5806,  ..., -1.3610, -0.3007,  1.4981],
          ...,
          [ 0.8143, -0.7950, -0.4548,  ..., -0.3649, -0.5259,  0.9135],
          [-2.2725, -0.4623, -0.6641,  ..., -1.1984, -0.0494,  0.3322],
          [ 0.9917,  0.2498, -0.9611,  ...,  1.6349, -2.0049,  0.6225]]]))

In [32]:
attn = scaled_dot_product_attention(Q, K, V)
attn.shape

torch.Size([1, 2, 5, 16])

In [33]:
attn = rearrange(attn, 'batch num_heads seq_len d_k -> batch seq_len (num_heads d_k)')
attn.shape

torch.Size([1, 5, 32])

In [36]:
Wo = rearrange(Wo, 'b d_m d_m2 -> b d_m2 d_m')
Wo

tensor([[[-0.8678,  0.7856, -0.8657,  ...,  0.8143, -2.2725,  0.9917],
         [-2.0687, -0.2031, -1.8815,  ..., -0.7950, -0.4623,  0.2498],
         [ 0.8783, -0.2220,  0.5806,  ..., -0.4548, -0.6641, -0.9611],
         ...,
         [ 0.8992,  0.9088, -1.3610,  ..., -0.3649, -1.1984,  1.6349],
         [ 0.0447,  0.2186, -0.3007,  ..., -0.5259, -0.0494, -2.0049],
         [-0.4222, -1.1714,  1.4981,  ...,  0.9135,  0.3322,  0.6225]]])

In [38]:
out = attn @ Wo
out, out.shape

(tensor([[[-3.1834e+01,  3.2027e+01, -4.9721e+01, -4.0739e+01,  8.4392e+00,
            4.3271e+01, -3.2285e+01, -5.8903e+01, -2.8362e+01, -4.1109e+01,
            2.0669e+01,  2.7788e+01, -6.6300e+00,  3.0740e+01,  1.0931e+01,
            1.3829e+01, -4.4346e+01,  1.6150e+01,  5.4513e+01,  3.3871e+01,
           -2.5172e+01, -4.7063e+00,  1.5089e+01,  1.3172e+00,  8.5122e+00,
            3.0809e+01, -3.0952e+01,  1.8318e+01,  1.3535e+01,  3.1171e+01,
            3.5273e+01,  5.4861e+01],
          [ 1.1758e+01, -4.0556e+00,  4.7160e+01, -4.5974e+00,  4.0172e+01,
            1.8026e+01,  6.0452e+00, -4.3881e+00, -2.5660e+01,  1.3755e+01,
            2.4579e+01, -1.6635e+01,  3.9836e+00, -2.0592e+01,  5.5722e+00,
           -1.6315e+01, -4.8035e+00, -8.0899e+00, -4.2966e+01, -1.4127e+01,
           -1.6538e-02, -4.6957e+01,  1.8929e+00, -2.5536e+01,  4.6196e+01,
            4.0569e+00,  6.8183e+00,  1.4809e+01,  2.2267e+01,  1.9145e+00,
           -3.4756e+00, -1.2536e+01],
          [-

In [ ]:
def causal_multihead_attention(x: torch.Tensor, d_model: int, num_heads: int) -> torch.Tensor:
    pass
    
def multi_head(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, num_heads: int, mask: torch.Tensor | None = None) -> torch.Tensor:
    pass

